## Data Loading

In [ ]:
from IPython.display import display
from PIL import Image
from datasets import Dataset
TRAIN_TEXT = "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

# show example
display(hint_description[7]['icons'])
print(hint_description[7]['description'])

In [ ]:
validation_data = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/takehome_validation")
validation_data

## Implement keyword guesser

Internet access is allowed in Bohrium, so contestants could download pre-trained models from huggingface. However, since the servers are hosted on mainland China, they are subject to internet restrictions that blocks access to sources like huggingface. To circumvent this, we can use hosted mirror servers(which we'll also provide at the on-site round). For huggingface, you can set os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  to use a huggingface mirror that's accessible from Bohrium's server.

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [ ]:
!ls /bohr/model-mioz/v6

In [ ]:
from sentence_transformers import SentenceTransformer
import transformers
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("KurmaAI/AQUA-1B")
# model = AutoModelForCausalLM.from_pretrained("KurmaAI/AQUA-1B")
device = torch.device('cuda')

model = SentenceTransformer("/bohr/model-mioz/v6/sentence-t5-large")

# transfer model
model = model.to(device)
# pipe(messages)
embeddings = model.encode([
    'hello world',
    'fun and games'
])

print(f"Embedding shape: {embeddings.shape}")

In [ ]:
# prompt = "Options: sunflower, credit card, dinosaur, key, sundial, lawyer, doorbell, trash can, crab, xylophone, queen, ambulance, space station, wallet, market, orchestra, chocolate, zipper, rhinoceros, fashion, butterfly, truck, palm tree, cake, radio, seal, mailbox, magnifying glass, prison, polar bear, mouse, alumunium foil, harmonica, shell, boxers, tricycle, peacock, kettle, mountain, harbor, coffee, fireworks, pie, gravity, teacher, museum, bedroom, robe, sunscreen, robot, piano, baker, plankton, scarf, bee, mosquito, accountant, umbrella, janitor, thief, parrot, koala, refrigerator, drone, dining room, soap, whistle, bicycle, train tracks, penguin, octopus, hula hoop, ice skates, nightmare, diving suit, horseshoe, dynamite, surfboard, toaster, gloves, broom, postal worker, lipstick, sewing machine, salad, dam, pool, fertilizer, shovel, speaker, seahorse, submarine, pig, mango, fire station, ping-pong, hotel, carpet, shoes, parachute \n \
# Hints: fauna, animal, water, liquid, aquatic, earth, ground, grey, fast, race. \n \
# The hints are related to the answer in order of importance, first one is the most important and last is the least important. Output the 10 most probable options for the hints."
# # Tokenize input
# inputs = tokenizer(prompt, return_tensors="pt").to(device)

# # Generate output (you can adjust max_length, temperature, etc.)
# outputs = model.generate(
#     inputs["input_ids"],
#     max_length=1500,
#     num_return_sequences=1,
#     # do_sample=True,  # for creativity
#     temperature=0.7  # controls randomness
# )

# # Decode and print
# generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(generated_text)

In [ ]:
def hints_to_sentence(hints: list[int]) -> str:
  sentence = "The following hints at our target word:\n<HINT_PRIMARY>\n"
  for i, hint in enumerate(hints):
    sentence += f"{hint_description[hint]['description']}"
    if i == 0:
      sentence += "\n</HINT_PRIMARY>\n<HINT>\n"
    elif i < len(hints) - 1:
      sentence += "\n</HINT>\n<HINT>\n"
    else:
      sentence += "\n</HINT>"
  return sentence

def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

print(hints_to_sentence([1, 2, 3]))

In [ ]:
# Fine-tune the model

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

import os
os.environ["WANDB_DISABLED"] = "true"

train_examples = []
for val in validation_data:
  train_examples.append(InputExample(texts=[hints_to_sentence(val['hints']), choice_to_doc(val['label'])], label=1))

# Create DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=1)

# Define loss function
train_loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=3,
    output_path='./model',
    # optimizer_params={'lr': 1e-7},
    save_best_model=True,
    show_progress_bar=True
)

In [ ]:
ft_model_loaded = SentenceTransformer("/model") # Load fine-tuned model

def find_most_similar(query, sentences, model, top_k=10):
    # Encode query and sentences
    print(query)
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calculate similarities
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Get top-k most similar
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results

def guess_words(hints: list[int], choices: list[str]) -> list[str]:
  query = hints_to_sentence(hints)
  results = find_most_similar(query, choices, ft_model_loaded)
  return [result['sentence'] for result in results]

In [ ]:
guess_words([4, 34, 60], 
            ['alumunium foil', 'train tracks', 'octopus', 'mountain', 'drill', 'queen', 'bull', 'sword', 'strawberry', 'magician', 'bee', 'cucumber', 'truck', 'well', 'plankton', 'window', 'crossword', 'horse', 'tiger', 'ambulance', 'gravity', 'sausage', 'mole', 'yogurt', 'doghouse', 'cyclist', 'ostrich', 'lobster', 'doorbell', 'harbor', 'binoculars', 'monkey', 'milk', 'trash can', 'bicycle', 'sewing machine', 'firefighter', 'firefighters', 'cave', 'prison', 'calculator', 'whale', 'farmer', 'eagle', 'fish', 'rocket', 'umbrella', 'diving suit', 'submarine', 'wheel', 'perfume', 'board game', 'seal', 'chameleon', 'river', 'fertilizer', 'nightmare', 'cabbage', 'hospital', 'market', 'mechanic', 'theater', 'ping-pong', 'jellyfish', 'cake', 'cheese', 'sundial', 'vitamins', 'dinosaur', 'holiday', 'toothpaste', 'key', 'porch', 'fisherman', 'surfboard', 'mosque', 'charger', 'straw', 'sled', 'rhinoceros', 'parachute', 'socks', 'ladder', 'printer', 'cabin', 'toothbrush', 'chilli pepper', 'police station', 'shorts', 'chocolate', 'cafeteria', 'fireworks', 'garden hose', 'fox', 'microphone', 'toaster', 'palace', 'cockroach', 't-shirt', 'meteorite'])

In [ ]:
import math

def score(guesses: list[str], gold: str):
    # Normalize to lowercase
    guesses = [g.lower() for g in guesses[:10]]
    gold = gold.lower()

    result = {
        "hits@10": 0.0,
        "ndcg@10": 0.0,
        "total_score": 0.0
    }

    if gold in guesses:
        rank = guesses.index(gold)
        result["hits@10"] = 1.0
        result["ndcg@10"] = 1.0 / math.log2(rank + 2)  # rank + 2 because index is 0-based
    else:
        result["hits@10"] = 0.0
        result["ndcg@10"] = 0.0

    result["total_score"] = 0.9 * result["hits@10"] + 0.1 * result["ndcg@10"]
    return result

print(score(['cat', 'dog', 'tree', 'flower', 'rock', 'water', 'fried rice', 'airplane', 'cactus', 'tiger'], gold='cactus'))

In [ ]:
from tqdm.notebook import tqdm

# score on validation set
guesses = []
total_scores = 0.0
for example in tqdm(validation_data):
    guesses.append(guess_words(example['hints'], example['options']))
    print(example['options'])

    total_scores += score(guesses[-1], example['label'])['total_score']


print(f"Average validation score: {total_scores / len(validation_data)}")

## Model Submission

In [ ]:
model_code = """
from sentence_transformers import SentenceTransformer
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

TRAIN_TEXT = "/bohr/train-7ul9/v2"
hint_description = Dataset.load_from_disk(TRAIN_TEXT + "/dataset/hint_descriptions")
hint_description = {
    x['ID']: {'description': x['Description'], 'icons': x['image']}
    for x in hint_description
}

model = SentenceTransformer("./model")

def hints_to_sentence(hints: list[int]) -> str:
  sentence = ""

  # fs_hint = ""
  for i, hint in enumerate(hints):
    h = str(hint_description[hint]['description'].lower())

    sentence += h + "\\n"

  sentence = sentence.replace('\\n', " ")
  sentence = sentence.replace('-', ' ')
  return sentence
  
def choice_to_doc(choice:str)->str:
  return f"Our target word: {choice}"

def find_most_similar(query, sentences, model, top_k=10):
    # Encode query and sentences
    query_embedding = model.encode([query])
    sentence_embeddings = model.encode(sentences)

    # Calculate similarities
    similarities = cosine_similarity(query_embedding, sentence_embeddings)[0]

    # Get top-k most similar
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'sentence': sentences[idx],
            'similarity': similarities[idx]
        })

    return results

def guess_words(hints: list[int], choices: list[str]) -> list[str]:
  query = hints_to_sentence(hints)
  results = find_most_similar(query, choices, model)
  return [result['sentence'] for result in results]
"""

with open("submission_model.py", "w") as f:
  f.write(model_code)

print("Inference code written to submission_model.py")

In [ ]:
import shutil
import os
import tempfile

# Create a temporary directory with your desired structure
with tempfile.TemporaryDirectory() as temp_dir:
    # Copy files to temp directory
    shutil.copy('submission_model.py', temp_dir)
    shutil.copytree('./model', os.path.join(temp_dir, 'model'))
    
    # Create the zip
    shutil.make_archive('submission', 'zip', temp_dir)